In [9]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
TOKEN = os.getenv("TINKOFF_TOKEN")

BASE_URL = "https://invest-public-api.tinkoff.ru/rest"
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

print(f"Токен загружен: {'✅' if TOKEN else '❌'}")

Токен загружен: ✅


In [11]:
TICKERS = ["SBER", "GAZP", "LKOH", "GMKN", "NVTK",
           "TATN", "MGNT", "ROSN", "MTSS", "ALRS",
           "CHMF", "NLMK", "PIKK", "VTBR", "PHOR"]

uid_map = {}

for ticker in TICKERS:
    resp = requests.post(
        f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/FindInstrument",
        headers=HEADERS,
        json={"query": ticker, "instrumentKind": "INSTRUMENT_TYPE_SHARE", "apiTradeAvailableFlag": True}
    )
    data = resp.json()
    for inst in data.get("instruments", []):
        if inst.get("ticker") == ticker:
            uid_map[ticker] = inst["uid"]
            print(f"{ticker} → {inst['uid']}")
            break

print(f"\nНайдено: {len(uid_map)}/{len(TICKERS)}")

SBER → e6123145-9665-43e0-8413-cd61b8aa9b13
GAZP → 962e2a95-02a9-4171-abd7-aa198dbe643a
LKOH → 02cfdf61-6298-4c0f-a9ca-9cabc82afaf3
GMKN → 509edd0c-129c-4ee2-934d-7f6246126da1
NVTK → 0da66728-6c30-44c4-9264-df8fac2467ee
TATN → 88468f6c-c67a-4fb4-a006-53eed803883c
MGNT → ca845f68-6c43-44bc-b584-330d2a1e5eb7
ROSN → fd417230-19cf-4e7b-9623-f7c9ca18ec6b
MTSS → cd8063ad-73ad-4b31-bd0d-93138d9e99a2
ALRS → 30817fea-20e6-4fee-ab1f-d20fc1a1bb72
CHMF → fa6aae10-b8d5-48c8-bbfd-d320d925d096
NLMK → 161eb0d0-aaac-4451-b374-f5d0eeb1b508
PIKK → 03d5e771-fc10-438e-8892-85a40733612d
VTBR → 8e2b0325-0292-4654-8a18-4f63ed3b0e09
PHOR → 9978b56f-782a-4a80-a4b1-a48cbecfd194

Найдено: 15/15


In [12]:
ticker_to_asset_uid = {}

for ticker, inst_uid in uid_map.items():
    resp = requests.post(
        f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetInstrumentBy",
        headers=HEADERS,
        json={"idType": "INSTRUMENT_ID_TYPE_UID", "id": inst_uid}
    )
    data = resp.json()
    asset_uid = data.get("instrument", {}).get("assetUid", "")
    ticker_to_asset_uid[ticker] = asset_uid
    print(f"{ticker} → asset_uid: {asset_uid}")

print(f"\nНайдено: {len(ticker_to_asset_uid)}")

SBER → asset_uid: 40d89385-a03a-4659-bf4e-d3ecba011782
GAZP → asset_uid: bfc8184d-9562-4ea2-87dd-be6e76dc1279
LKOH → asset_uid: f898047a-dac8-4717-a40f-7f11211139ef
GMKN → asset_uid: ba7b5a1b-8515-4134-8c26-b69a2372fe82
NVTK → asset_uid: 2d850de1-466d-443d-b7fd-e78bf41146fb
TATN → asset_uid: 9da0b54d-f8dd-4e75-9081-ec0b8257775c
MGNT → asset_uid: 4833e124-265f-4879-adc8-58bdf983f54e
ROSN → asset_uid: 47ba7e01-8f52-4723-90e5-3ea379b9ccae
MTSS → asset_uid: eb85ff63-713d-4ace-b4c8-922c9a5fc812
ALRS → asset_uid: f924d7df-b6d0-4e36-9bf7-f38aaac2ce53
CHMF → asset_uid: e72889a3-21db-4b2e-90bc-045ec3d28d04
NLMK → asset_uid: fb0fb8a8-fcd5-4f78-b5aa-73b3be9f4454
PIKK → asset_uid: c14ab000-c378-432d-9686-1055170a4c3d
VTBR → asset_uid: 94f18cc6-9e01-4db6-9d73-8c3ba6ab9655
PHOR → asset_uid: 5a3d1efd-f8a0-478e-a10e-bb7f990f9c87

Найдено: 15


In [13]:
asset_uids = list(ticker_to_asset_uid.values())

resp = requests.post(
    f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetAssetFundamentals",
    headers=HEADERS,
    json={"assets": asset_uids}
)
data = resp.json()

records = []
asset_uid_to_ticker = {v: k for k, v in ticker_to_asset_uid.items()}

for f in data.get("fundamentals", []):
    ticker = asset_uid_to_ticker.get(f.get("assetUid", ""), "UNKNOWN")
    records.append({
        "ticker":            ticker,
        "pe_ratio":          f.get("peRatioTtm"),
        "pb_ratio":          f.get("priceToBookTtm"),
        "ps_ratio":          f.get("priceToSalesTtm"),
        "ev_ebitda":         f.get("evToEbitdaMrq"),
        "roe":               f.get("roe"),
        "roa":               f.get("roa"),
        "net_margin":        f.get("netMarginMrq"),
        "debt_to_equity":    f.get("totalDebtToEquityMrq"),
        "debt_to_ebitda":    f.get("totalDebtToEbitdaMrq"),
        "net_debt_ebitda":   f.get("netDebtToEbitda"),
        "current_ratio":     f.get("currentRatioMrq"),
        "div_yield":         f.get("dividendYieldDailyTtm"),
        "revenue_growth_1y": f.get("oneYearAnnualRevenueGrowthRate"),
        "revenue_growth_3y": f.get("threeYearAnnualRevenueGrowthRate"),
        "revenue_growth_5y": f.get("fiveYearAnnualRevenueGrowthRate"),
        "eps_ttm":           f.get("epsTtm"),
        "beta":              f.get("beta"),
        "market_cap":        f.get("marketCapitalization"),
    })

df_fund = pd.DataFrame(records)
print(df_fund.to_string())

   ticker  pe_ratio  pb_ratio  ps_ratio  ev_ebitda     roe    roa  net_margin  debt_to_equity  debt_to_ebitda  net_debt_ebitda  current_ratio  div_yield  revenue_growth_1y  revenue_growth_3y  revenue_growth_5y  eps_ttm  beta    market_cap
0    SBER      4.00      0.82      0.63       0.00   22.19   2.70        0.00            0.00             0.0             0.00            0.0      11.02              25.20              33.00                0.0    79.02  0.52  6.827736e+12
1    GAZP      2.14      0.17      0.30       0.00    7.95   4.71       13.91           35.17             0.0             0.00            0.0       0.00              25.44               1.52                0.0    60.98  0.97  3.091287e+12
2    LKOH      7.19      0.70      0.50       2.70    8.66   6.42        6.95            6.14             0.0            -0.10            0.0      16.49               8.74              -2.96                0.0   791.74  0.75  3.941713e+12
3    GMKN     11.66      2.29      2.09     